In [12]:
from gurobipy import *
model = Model('Benders decomposition')

""" create decision variables """
y=model.addVar(lb=0,ub=1000,vtype=GRB.INTEGER,name='y')
x = {}
for i in range(1,11):
    x[i] = model.addVar(lb=0,ub=100,vtype=GRB.CONTINUOUS,name='x_'+str(i))

""" set objective function """
obj = LinExpr()
obj.addTerms(1.045, y)
for i in range(1,11):
    obj.addTerms(1+0.01*i,x[i])
model.setObjective(obj, GRB.MAXIMIZE)

""" add constraints """
lhs = LinExpr()
lhs.addTerms(1,y)
for i in range(1,11):
    lhs.addTerms(1,x[i])
model.addConstr(lhs <= 1000, name='budget')

model.optimize()
print('\n\n\n')
print('Obj:', model.ObjVal)
print('Saving account : ', y.x)
for i in range(1,11):
    if(x[i].x > 0):
        print('Fund ID {}: amount: {}'.format(i,x[i].x))


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 1 rows, 11 columns and 11 nonzeros
Model fingerprint: 0x8037ad5f
Variable types: 10 continuous, 1 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+02, 1e+03]
  RHS range        [1e+03, 1e+03]
Found heuristic solution: objective 1045.0000000
Presolve time: 0.00s
Presolved: 1 rows, 11 columns, 11 nonzeros
Variable types: 10 continuous, 1 integer (0 binary)

Root relaxation: objective 1.063000e+03, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0    1063.0000000 1063.0

In [18]:
from gurobipy import *

MP =Model("benders decomposition-MP")
y = MP.addVar(lb=0,ub=1000,vtype =GRB.INTEGER,name='y')
z = MP.addVar(lb=0, ub =GRB.INFINITY,vtype=GRB.INTEGER,name='z')
MP.setObjective(z, GRB.MAXIMIZE)

MP.optimize()
print('\n\n\n')
print('Obj:', MP.ObjVal)
print('z = %4.1f' % (z.x))
print('y = %4.1f' % (y.x))


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 0 rows, 2 columns and 0 nonzeros
Model fingerprint: 0x74c03ffd
Variable types: 0 continuous, 2 integer (0 binary)
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+03, 1e+03]
  RHS range        [0e+00, 0e+00]
Found heuristic solution: objective 1.000000e+30
Presolve removed 0 rows and 1 columns
Presolve time: 0.00s

Explored 0 nodes (0 simplex iterations) in 0.00 seconds (0.00 work units)
Thread count was 1 (of 20 available processors)

Solution count 1: 1e+30 
No other solutions better than 0

Model is unbounded
Best objective 1.000000000000e+30, best bound -, gap -




Obj: 1e+30
z = 1000000000000000019884624838656.0
y = -0.0


In [19]:
Dual_SP = Model('Dual SP')
y_bar = 0
alpha_0 = Dual_SP.addVar(lb=0,ub=GRB.INFINITY,vtype = GRB.CONTINUOUS,name='alpha_0')
alpha = {}
for i in range(1,11):
    alpha[i] = Dual_SP.addVar(lb=0,ub=GRB.INFINITY,vtype = GRB.CONTINUOUS,name='alpha_'+str(i))
obj = LinExpr()
obj.addTerms(1000-y_bar,alpha_0)
for i in range(1,11):
    obj.addTerms(100,alpha[i])
Dual_SP.setObjective(obj, GRB.MINIMIZE)

for i in range(1,11):
    Dual_SP.addConstr(alpha_0+alpha[i] >= 1+0.01*i,name = 'constrain_'+str(i))

Dual_SP.optimize()

print('\n\n\n')
print('Model status:', Dual_SP.status)
if(Dual_SP.status == 2):
    print('Obj:', Dual_SP.ObjVal)
    print('{} = {}'.format(alpha_0.varName, alpha_0.x))
    for i in range(1,11):
        if(alpha[i].x > 0):
            print('{} = {}'.format(alpha[i].varName, alpha[i].x))



Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 10 rows, 11 columns and 20 nonzeros
Model fingerprint: 0x3fdbd33c
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+02, 1e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve time: 0.00s
Presolved: 10 rows, 11 columns, 20 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   1.055000e+01   0.000000e+00      0s
      10    1.0550000e+03   0.000000e+00   0.000000e+00      0s

Solved in 10 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.055000000e+03




Model status: 2
Obj: 1055.0
alpha_0 = 0.0
alpha_1 = 1.01
alpha_2 = 1.02
alpha_3 = 1.03
alpha_4 = 1.04
alpha_5 = 1.05
alpha_6 = 1.06
alpha_7 = 1.07
al

In [20]:
from gurobipy import *

MP =Model("benders decomposition-MP")
y = MP.addVar(lb=0,ub=1000,vtype =GRB.INTEGER,name='y')
z = MP.addVar(lb=0, ub =GRB.INFINITY,vtype=GRB.INTEGER,name='z')
MP.setObjective(z, GRB.MAXIMIZE)
MP.addConstr(1.045*y+1055>=z,name='optimality cut 1')
MP.optimize()
print('\n\n\n')
print('Obj:', MP.ObjVal)
print('z = %4.1f' % (z.x))
print('y = %4.1f' % (y.x))

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 1 rows, 2 columns and 2 nonzeros
Model fingerprint: 0x64923304
Variable types: 0 continuous, 2 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+03, 1e+03]
  RHS range        [1e+03, 1e+03]
Found heuristic solution: objective 1055.0000000
Found heuristic solution: objective 2100.0000000
Presolve removed 1 rows and 2 columns
Presolve time: 0.00s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.01 seconds (0.00 work units)
Thread count was 1 (of 20 available processors)

Solution count 2: 2100 1055 

Optimal solution found (tolerance 1.00e-04)
Best objective 2.100000000000e+03, best bound 2.100000000000e+03

In [21]:
Dual_SP = Model('Dual SP')
y_bar = 1000
alpha_0 = Dual_SP.addVar(lb=0,ub=GRB.INFINITY,vtype = GRB.CONTINUOUS,name='alpha_0')
alpha = {}
for i in range(1,11):
    alpha[i] = Dual_SP.addVar(lb=0,ub=GRB.INFINITY,vtype = GRB.CONTINUOUS,name='alpha_'+str(i))
obj = LinExpr()
obj.addTerms(1000-y_bar,alpha_0)
for i in range(1,11):
    obj.addTerms(100,alpha[i])
Dual_SP.setObjective(obj, GRB.MINIMIZE)
for i in range(1,11):
    Dual_SP.addConstr(alpha_0+alpha[i] >= 1+0.01*i,name = 'constrain_'+str(i))
Dual_SP.optimize()
print('\n\n\n')
print('Model status:', Dual_SP.status)
if(Dual_SP.status == 2):
    print('Obj:', Dual_SP.ObjVal)
    print('{} = {}'.format(alpha_0.varName, alpha_0.x))
    for i in range(1,11):
        if(alpha[i].x > 0):
            print('{} = {}'.format(alpha[i].varName, alpha[i].x))



Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 10 rows, 11 columns and 20 nonzeros
Model fingerprint: 0x23db86f6
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+02, 1e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 10 rows and 11 columns
Presolve time: 0.00s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  0.000000000e+00




Model status: 2
Obj: 0.0
alpha_0 = 1.1


In [24]:
from gurobipy import *

MP =Model("benders decomposition-MP")
y = MP.addVar(lb=0,ub=1000,vtype =GRB.INTEGER,name='y')
z = MP.addVar(lb=0, ub =GRB.INFINITY,vtype=GRB.INTEGER,name='z')
MP.setObjective(z, GRB.MAXIMIZE)
MP.addConstr(1100-0.055*y>=z,name='optimality cut 2')
MP.addConstr(1.045*y+1055>=z,name='optimality cut 1')
MP.optimize()
print('\n\n\n')
print('Obj:', MP.ObjVal)
print('z = %4.1f' % (z.x))
print('y = %4.1f' % (y.x))


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 2 rows, 2 columns and 4 nonzeros
Model fingerprint: 0xd28fe636
Variable types: 0 continuous, 2 integer (0 binary)
Coefficient statistics:
  Matrix range     [6e-02, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+03, 1e+03]
  RHS range        [1e+03, 1e+03]
Found heuristic solution: objective 1055.0000000
Presolve time: 0.00s
Presolved: 2 rows, 2 columns, 4 nonzeros
Variable types: 0 continuous, 2 integer (0 binary)

Root relaxation: objective 1.097000e+03, 0 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0    1097.0000000 1097.00000  

In [25]:
Dual_SP = Model('Dual SP')
y_bar =44
alpha_0 = Dual_SP.addVar(lb=0,ub=GRB.INFINITY,vtype = GRB.CONTINUOUS,name='alpha_0')
alpha = {}
for i in range(1,11):
    alpha[i] = Dual_SP.addVar(lb=0,ub=GRB.INFINITY,vtype = GRB.CONTINUOUS,name='alpha_'+str(i))
obj = LinExpr()
obj.addTerms(1000-y_bar,alpha_0)
for i in range(1,11):
    obj.addTerms(100,alpha[i])
Dual_SP.setObjective(obj, GRB.MINIMIZE)
for i in range(1,11):
    Dual_SP.addConstr(alpha_0+alpha[i] >= 1+0.01*i,name = 'constrain_'+str(i))
Dual_SP.optimize()
print('\n\n\n')
print('Model status:', Dual_SP.status)
if(Dual_SP.status == 2):
    print('Obj:', Dual_SP.ObjVal)
    print('{} = {}'.format(alpha_0.varName, alpha_0.x))
    for i in range(1,11):
        if(alpha[i].x > 0):
            print('{} = {}'.format(alpha[i].varName, alpha[i].x))

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 10 rows, 11 columns and 20 nonzeros
Model fingerprint: 0x7828e95d
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+02, 1e+03]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve time: 0.00s
Presolved: 10 rows, 11 columns, 20 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   1.055000e+01   0.000000e+00      0s
      10    1.0105600e+03   0.000000e+00   0.000000e+00      0s

Solved in 10 iterations and 0.01 seconds (0.00 work units)
Optimal objective  1.010560000e+03




Model status: 2
Obj: 1010.5600000000001
alpha_0 = 1.01
alpha_2 = 0.010000000000000009
alpha_3 = 0.020000000000000018
alpha_4 = 0.030000000000000027
a

In [28]:
from gurobipy import *

MP =Model("benders decomposition-MP")
y = MP.addVar(lb=0,ub=1000,vtype =GRB.INTEGER,name='y')
z = MP.addVar(lb=0, ub =GRB.INFINITY,vtype=GRB.INTEGER,name='z')
MP.setObjective(z, GRB.MAXIMIZE)
MP.addConstr(1100-0.055*y>=z,name='optimality cut 2')
MP.addConstr(1.045*y+1055>=z,name='optimality cut 1')
MP.addConstr(1055+0.035*y>=z,name='optimality cut 3')
MP.optimize()
print('\n\n\n')
print('Obj:', MP.ObjVal)
print('z = %4.1f' % (z.x))
print('y = %4.1f' % (y.x))


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 3 rows, 2 columns and 6 nonzeros
Model fingerprint: 0x73bc731a
Variable types: 0 continuous, 2 integer (0 binary)
Coefficient statistics:
  Matrix range     [4e-02, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+03, 1e+03]
  RHS range        [1e+03, 1e+03]
Found heuristic solution: objective 1055.0000000
Presolve removed 1 rows and 0 columns
Presolve time: 0.00s
Presolved: 2 rows, 2 columns, 4 nonzeros
Variable types: 0 continuous, 2 integer (0 binary)

Root relaxation: objective 1.072000e+03, 0 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0       

In [29]:
Dual_SP = Model('Dual SP')
y_bar =509
alpha_0 = Dual_SP.addVar(lb=0,ub=GRB.INFINITY,vtype = GRB.CONTINUOUS,name='alpha_0')
alpha = {}
for i in range(1,11):
    alpha[i] = Dual_SP.addVar(lb=0,ub=GRB.INFINITY,vtype = GRB.CONTINUOUS,name='alpha_'+str(i))
obj = LinExpr()
obj.addTerms(1000-y_bar,alpha_0)
for i in range(1,11):
    obj.addTerms(100,alpha[i])
Dual_SP.setObjective(obj, GRB.MINIMIZE)
for i in range(1,11):
    Dual_SP.addConstr(alpha_0+alpha[i] >= 1+0.01*i,name = 'constrain_'+str(i))
Dual_SP.optimize()
print('\n\n\n')
print('Model status:', Dual_SP.status)
if(Dual_SP.status == 2):
    print('Obj:', Dual_SP.ObjVal)
    print('{} = {}'.format(alpha_0.varName, alpha_0.x))
    for i in range(1,11):
        if(alpha[i].x > 0):
            print('{} = {}'.format(alpha[i].varName, alpha[i].x))

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 10 rows, 11 columns and 20 nonzeros
Model fingerprint: 0xd04953c6
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+02, 5e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve time: 0.00s
Presolved: 10 rows, 11 columns, 20 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   1.055000e+01   0.000000e+00      0s
       5    5.3046000e+02   0.000000e+00   0.000000e+00      0s

Solved in 5 iterations and 0.01 seconds (0.00 work units)
Optimal objective  5.304600000e+02




Model status: 2
Obj: 530.46
alpha_0 = 1.06
alpha_7 = 0.010000000000000009
alpha_8 = 0.020000000000000018
alpha_9 = 0.030000000000000027
alpha_10 = 0.0

In [30]:
from gurobipy import *

MP =Model("benders decomposition-MP")
y = MP.addVar(lb=0,ub=1000,vtype =GRB.INTEGER,name='y')
z = MP.addVar(lb=0, ub =GRB.INFINITY,vtype=GRB.INTEGER,name='z')
MP.setObjective(z, GRB.MAXIMIZE)
MP.addConstr(1100-0.055*y>=z,name='optimality cut 2')
MP.addConstr(1.045*y+1055>=z,name='optimality cut 1')
MP.addConstr(1055+0.035*y>=z,name='optimality cut 3')
MP.addConstr(1070-0.015*y>=z,name='optimality cut 4')
MP.optimize()
print('\n\n\n')
print('Obj:', MP.ObjVal)
print('z = %4.1f' % (z.x))
print('y = %4.1f' % (y.x))


Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 4 rows, 2 columns and 8 nonzeros
Model fingerprint: 0x940425f1
Variable types: 0 continuous, 2 integer (0 binary)
Coefficient statistics:
  Matrix range     [1e-02, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+03, 1e+03]
  RHS range        [1e+03, 1e+03]
Found heuristic solution: objective 1055.0000000
Presolve removed 2 rows and 0 columns
Presolve time: 0.00s
Presolved: 2 rows, 2 columns, 4 nonzeros
Variable types: 0 continuous, 2 integer (0 binary)

Root relaxation: objective 1.065000e+03, 0 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0       

In [31]:
Dual_SP = Model('Dual SP')
y_bar =333
alpha_0 = Dual_SP.addVar(lb=0,ub=GRB.INFINITY,vtype = GRB.CONTINUOUS,name='alpha_0')
alpha = {}
for i in range(1,11):
    alpha[i] = Dual_SP.addVar(lb=0,ub=GRB.INFINITY,vtype = GRB.CONTINUOUS,name='alpha_'+str(i))
obj = LinExpr()
obj.addTerms(1000-y_bar,alpha_0)
for i in range(1,11):
    obj.addTerms(100,alpha[i])
Dual_SP.setObjective(obj, GRB.MINIMIZE)
for i in range(1,11):
    Dual_SP.addConstr(alpha_0+alpha[i] >= 1+0.01*i,name = 'constrain_'+str(i))
Dual_SP.optimize()
print('\n\n\n')
print('Model status:', Dual_SP.status)
if(Dual_SP.status == 2):
    print('Obj:', Dual_SP.ObjVal)
    print('{} = {}'.format(alpha_0.varName, alpha_0.x))
    for i in range(1,11):
        if(alpha[i].x > 0):
            print('{} = {}'.format(alpha[i].varName, alpha[i].x))

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 10 rows, 11 columns and 20 nonzeros
Model fingerprint: 0x5453b8ae
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+02, 7e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve time: 0.00s
Presolved: 10 rows, 11 columns, 20 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   1.055000e+01   0.000000e+00      0s
       7    7.1468000e+02   0.000000e+00   0.000000e+00      0s

Solved in 7 iterations and 0.01 seconds (0.00 work units)
Optimal objective  7.146800000e+02




Model status: 2
Obj: 714.6800000000001
alpha_0 = 1.04
alpha_5 = 0.010000000000000009
alpha_6 = 0.020000000000000018
alpha_7 = 0.030000000000000027
alp

In [32]:
from gurobipy import *

MP =Model("benders decomposition-MP")
y = MP.addVar(lb=0,ub=1000,vtype =GRB.INTEGER,name='y')
z = MP.addVar(lb=0, ub =GRB.INFINITY,vtype=GRB.INTEGER,name='z')
MP.setObjective(z, GRB.MAXIMIZE)
MP.addConstr(1100-0.055*y>=z,name='optimality cut 2')
MP.addConstr(1.045*y+1055>=z,name='optimality cut 1')
MP.addConstr(1055+0.035*y>=z,name='optimality cut 3')
MP.addConstr(1070-0.015*y>=z,name='optimality cut 4')
MP.addConstr(1061+0.005*y>=z,name='optimality cut 5')
MP.optimize()
print('\n\n\n')
print('Obj:', MP.ObjVal)
print('z = %4.1f' % (z.x))
print('y = %4.1f' % (y.x))

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 5 rows, 2 columns and 10 nonzeros
Model fingerprint: 0xfd9f33de
Variable types: 0 continuous, 2 integer (0 binary)
Coefficient statistics:
  Matrix range     [5e-03, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+03, 1e+03]
  RHS range        [1e+03, 1e+03]
Found heuristic solution: objective 1055.0000000
Presolve removed 2 rows and 0 columns
Presolve time: 0.00s
Presolved: 3 rows, 2 columns, 6 nonzeros
Variable types: 0 continuous, 2 integer (0 binary)

Root relaxation: objective 1.063000e+03, 0 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0      

In [33]:
Dual_SP = Model('Dual SP')
y_bar =466
alpha_0 = Dual_SP.addVar(lb=0,ub=GRB.INFINITY,vtype = GRB.CONTINUOUS,name='alpha_0')
alpha = {}
for i in range(1,11):
    alpha[i] = Dual_SP.addVar(lb=0,ub=GRB.INFINITY,vtype = GRB.CONTINUOUS,name='alpha_'+str(i))
obj = LinExpr()
obj.addTerms(1000-y_bar,alpha_0)
for i in range(1,11):
    obj.addTerms(100,alpha[i])
Dual_SP.setObjective(obj, GRB.MINIMIZE)
for i in range(1,11):
    Dual_SP.addConstr(alpha_0+alpha[i] >= 1+0.01*i,name = 'constrain_'+str(i))
Dual_SP.optimize()
print('\n\n\n')
print('Model status:', Dual_SP.status)
if(Dual_SP.status == 2):
    print('Obj:', Dual_SP.ObjVal)
    print('{} = {}'.format(alpha_0.varName, alpha_0.x))
    for i in range(1,11):
        if(alpha[i].x > 0):
            print('{} = {}'.format(alpha[i].varName, alpha[i].x))

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 10 rows, 11 columns and 20 nonzeros
Model fingerprint: 0x77e52f9a
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+02, 5e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve time: 0.00s
Presolved: 10 rows, 11 columns, 20 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   1.055000e+01   0.000000e+00      0s
       6    5.7570000e+02   0.000000e+00   0.000000e+00      0s

Solved in 6 iterations and 0.01 seconds (0.00 work units)
Optimal objective  5.757000000e+02




Model status: 2
Obj: 575.7
alpha_0 = 1.05
alpha_6 = 0.010000000000000009
alpha_7 = 0.020000000000000018
alpha_8 = 0.030000000000000027
alpha_9 = 0.040

In [34]:
from gurobipy import *

MP =Model("benders decomposition-MP")
y = MP.addVar(lb=0,ub=1000,vtype =GRB.INTEGER,name='y')
z = MP.addVar(lb=0, ub =GRB.INFINITY,vtype=GRB.INTEGER,name='z')
MP.setObjective(z, GRB.MAXIMIZE)
MP.addConstr(1100-0.055*y>=z,name='optimality cut 2')
MP.addConstr(1.045*y+1055>=z,name='optimality cut 1')
MP.addConstr(1055+0.035*y>=z,name='optimality cut 3')
MP.addConstr(1070-0.015*y>=z,name='optimality cut 4')
MP.addConstr(1061+0.005*y>=z,name='optimality cut 5')
MP.addConstr(1065-0.005*y>=z,name='optimality cut 6')
MP.optimize()
print('\n\n\n')
print('Obj:', MP.ObjVal)
print('z = %4.1f' % (z.x))
print('y = %4.1f' % (y.x))

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 6 rows, 2 columns and 12 nonzeros
Model fingerprint: 0x36a21c3d
Variable types: 0 continuous, 2 integer (0 binary)
Coefficient statistics:
  Matrix range     [5e-03, 1e+00]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e+03, 1e+03]
  RHS range        [1e+03, 1e+03]
Found heuristic solution: objective 1055.0000000
Presolve removed 1 rows and 0 columns
Presolve time: 0.00s
Presolved: 5 rows, 2 columns, 10 nonzeros
Variable types: 0 continuous, 2 integer (0 binary)

Root relaxation: objective 1.063000e+03, 1 iterations, 0.00 seconds (0.00 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0     

In [35]:
Dual_SP = Model('Dual SP')
y_bar =400
alpha_0 = Dual_SP.addVar(lb=0,ub=GRB.INFINITY,vtype = GRB.CONTINUOUS,name='alpha_0')
alpha = {}
for i in range(1,11):
    alpha[i] = Dual_SP.addVar(lb=0,ub=GRB.INFINITY,vtype = GRB.CONTINUOUS,name='alpha_'+str(i))
obj = LinExpr()
obj.addTerms(1000-y_bar,alpha_0)
for i in range(1,11):
    obj.addTerms(100,alpha[i])
Dual_SP.setObjective(obj, GRB.MINIMIZE)
for i in range(1,11):
    Dual_SP.addConstr(alpha_0+alpha[i] >= 1+0.01*i,name = 'constrain_'+str(i))
Dual_SP.optimize()
print('\n\n\n')
print('Model status:', Dual_SP.status)
if(Dual_SP.status == 2):
    print('Obj:', Dual_SP.ObjVal)
    print('{} = {}'.format(alpha_0.varName, alpha_0.x))
    for i in range(1,11):
        if(alpha[i].x > 0):
            print('{} = {}'.format(alpha[i].varName, alpha[i].x))

Gurobi Optimizer version 12.0.3 build v12.0.3rc0 (win64 - Windows 11+.0 (26200.2))

CPU model: 13th Gen Intel(R) Core(TM) i7-13650HX, instruction set [SSE2|AVX|AVX2]
Thread count: 14 physical cores, 20 logical processors, using up to 20 threads

Optimize a model with 10 rows, 11 columns and 20 nonzeros
Model fingerprint: 0x126ca48e
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+02, 6e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve time: 0.00s
Presolved: 10 rows, 11 columns, 20 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   1.055000e+01   0.000000e+00      0s
       7    6.4500000e+02   0.000000e+00   0.000000e+00      0s

Solved in 7 iterations and 0.01 seconds (0.00 work units)
Optimal objective  6.450000000e+02




Model status: 2
Obj: 645.0
alpha_0 = 1.04
alpha_5 = 0.010000000000000009
alpha_6 = 0.020000000000000018
alpha_7 = 0.030000000000000027
alpha_8 = 0.040

In [37]:
print('\n\n\n  ==== Solution of SP primal ===== ')
cnt = 1
for con in Dual_SP.getConstrs():
    print('x {} = {}'.format(cnt, con.Pi))
    cnt += 1




  ==== Solution of SP primal ===== 
x 1 = 0.0
x 2 = 0.0
x 3 = 0.0
x 4 = 0.0
x 5 = 100.0
x 6 = 100.0
x 7 = 100.0
x 8 = 100.0
x 9 = 100.0
x 10 = 100.0
